# RegRAG from scratch

The whole RegRAG project written as **simple functions in one notebook**. It does not import the
`regrag` package; every step is written out below.

| Step | What it does | Same idea as |
|---|---|---|
| 1. Settings | folders, models, sizes | `config.py` |
| 2. Read & split PDFs | PDF → clause-sized chunks | `ingest.py` |
| 3. Build indexes | vector index + keyword index | `index.py` |
| 4. Understand the question | which state(s)? | `query.py` |
| 5. Search | find the best 5 clauses | `retrieve.py` |
| 6. Check amendments | flag rules that a newer order changed | `amendments.py` |
| 7. Write the answer | cited answer, or decline | `generate.py` |
| 8. Full pipeline | steps 4–7 in one function | `pipeline.py` |
| 9. Agent | same, plus a self-check and one retry | `graph.py` |
| 10. Evaluation | score 34 hand-written questions | `eval/run_eval.py` |

**Models:** answers are written by **`openai/gpt-oss-120b` on Groq** (as in the class lab). Embeddings
(`nomic-embed-text` via Ollama) and reranking (a small cross-encoder) run locally.

**Before running:**
- Start Ollama (`ollama serve`), which is needed for the embeddings.
- Have `GROQ_API_KEY` in the project's `.env` file. If it's missing, you'll be asked for it.
- Choose the kernel **Python (RegRAG .venv)**.

Building the vector index takes about 10 minutes the first time; it's saved and reused afterwards.
Cells marked ⏱ call the LLM (a few seconds each on Groq).

## Setup

In [1]:
import getpass
import json
import logging
import os
import re
import warnings
from collections import Counter, defaultdict
from pathlib import Path

# Hide library warnings and progress bars so the outputs stay readable
warnings.filterwarnings("ignore")
logging.getLogger("pypdf").setLevel(logging.CRITICAL)
os.environ["TQDM_DISABLE"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

from dotenv import load_dotenv
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# The project folder (works whether the notebook is in the project folder or in notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Project folder:", ROOT)

# Groq API key: read from the project's .env file, or ask for it
load_dotenv(ROOT / ".env")
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ API Key: ")

Project folder: C:\Users\risha\OneDrive\Desktop\Mini Project


## 1. Settings

Everything you might want to change is in one place.

In [2]:
RAW_DIR = ROOT / "data" / "raw"                   # the regulation PDFs
MANIFEST_FILE = ROOT / "data" / "manifest.json"   # list of documents with state, date, status
CHROMA_DIR = ROOT / "data" / "chroma_notebook"    # this notebook's own vector index

LLM_MODEL = "openai/gpt-oss-120b"                        # writes the answers (via Groq)
EMBED_MODEL = "nomic-embed-text"                         # turns text into vectors (via Ollama)
RERANK_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"    # re-scores search results

MIN_CHARS = 250       # smaller chunks are merged into the previous one
MAX_CHARS = 1800      # bigger chunks are split
SEARCH_K = 25         # results taken from each search
TOP_N = 5             # clauses given to the LLM
ABSTAIN_SCORE = -6.0  # if the best reranker score is below this, decline without calling the LLM

manifest = json.loads(MANIFEST_FILE.read_text(encoding="utf-8"))["documents"]
print(len(manifest), "documents in the manifest")

13 documents in the manifest


## 2. Read the PDFs and split them into clauses

Regulations are numbered (`6`, `6.2`, `7.3`), so we cut the text at clause numbers instead of every N characters.

The tricky part is that **not every numbered line is a clause**. Table rows (`1 Registration…`) and quantities
(`1 MW capacity`) also start with numbers. The rule that separates them: **real clause numbers only move
forward, in small steps**, while table rows keep restarting at 1.

In [3]:
def read_pdf_lines(pdf_path):
    """Return a list of (page_number, line), without page numbers, headers and tables of contents."""
    pages = []
    for page in PdfReader(pdf_path).pages:
        text = page.extract_text() or ""
        lines = [" ".join(line.split()) for line in text.splitlines()]   # tidy the spaces
        pages.append([line for line in lines if line])

    # A line that appears on many pages is a running header or footer
    counts = Counter(line for page in pages for line in set(page))
    repeated = {line for line, n in counts.items() if len(pages) >= 4 and n >= max(3, 0.4 * len(pages))}

    result = []
    for page_number, page in enumerate(pages, start=1):
        for line in page:
            if line in repeated:
                continue
            if re.fullmatch(r"(page\s*)?\d+(\s*of\s*\d+)?", line, re.IGNORECASE):   # "Page 3 of 32", "17"
                continue
            if "....." in line or "……" in line:                                       # table of contents
                continue
            result.append((page_number, line))
    return result

In [4]:
HEADING = re.compile(r"^(\d{1,2}(?:\.\d{1,2}){0,2})(\.?)\s+(\S.*)$")   # "6.2 The maximum..."
UNIT_WORDS = {"kw", "mw", "kva", "kwh", "rs", "rs.", "inr", "days", "months", "years", "per", "of", "to", "and", "or"}


def is_new_clause(number, has_dot, text, current, is_amendment, needs_dot):
    """Decide whether a numbered line starts a new clause."""
    if text.split()[0].lower().rstrip(",:;") in UNIT_WORDS:     # "1 MW capacity" is a quantity
        return False
    if not (text[0].isupper() or text[0] in "\"'(“"):           # "2 days from receipt" is mid-sentence
        return False
    major = int(number.split(".")[0])
    if major == 0 or major > 80:
        return False
    if current is None:                                         # the first clause is 1, 2 or 3
        return major <= 3
    if "." in number:                                           # a sub-clause like 6.2
        if is_amendment:
            return major == current     # amendments QUOTE old clauses; keep the quote inside the paragraph
        return current <= major <= current + 3
    if needs_dot and not has_dot:       # in "4. Title" documents, "7 Signing of" is a table row
        return False
    return current < major <= current + 2                       # numbers move forward in small steps


def split_into_clauses(lines, is_amendment=False, needs_dot=False):
    """Group lines into clauses: [{clause, section, page, lines}]."""
    clauses = [{"clause": "", "section": "", "page": 1, "lines": []}]   # the first one collects the preamble
    current = None        # number of the current top-level clause
    titles = {}           # top-level clause number -> its title, e.g. "6" -> "Eligible Consumer..."
    for page, line in lines:
        match = HEADING.match(line)
        if match and is_new_clause(match.group(1), match.group(2) == ".", match.group(3),
                                   current, is_amendment, needs_dot):
            number = match.group(1)
            major = number.split(".")[0]
            current = int(major)
            if "." not in number:
                titles[major] = match.group(3)[:80]
            clauses.append({"clause": number, "section": titles.get(major, ""), "page": page, "lines": [line]})
        else:
            clauses[-1]["lines"].append(line)
    return [c for c in clauses if c["lines"]]

In [5]:
AMEND_WORDS = ["substitut", "amend", "insert", "omit", "delet", "replac", "increas", "revis", "modif"]


def amended_clauses(text):
    """Clause numbers this text changes, e.g. 'Regulation 7 ... shall be substituted' -> ['7']."""
    found = set()
    for match in re.finditer(r"(?:[Rr]egulations?|[Cc]lauses?)\s+(\d{1,2}(?:\.\d{1,2})*)(?!\d)", text):
        nearby = text[max(0, match.start() - 120): match.end() + 120].lower()
        if any(word in nearby for word in AMEND_WORDS):     # only count it if an amending verb is close by
            found.add(match.group(1))
    return sorted(found)


splitter = RecursiveCharacterTextSplitter(chunk_size=MAX_CHARS, chunk_overlap=100)


def make_chunks(clauses, meta):
    """Turn clauses into chunks: merge tiny ones, split huge ones, attach the document's details."""
    pieces = []
    for clause in clauses:
        text = "\n".join(clause["lines"])
        if pieces and len(text) < MIN_CHARS:          # a bare heading: add it to the previous chunk
            pieces[-1]["text"] += "\n" + text
        else:
            pieces.append({**clause, "text": text})

    chunks = []
    for piece in pieces:
        for part in splitter.split_text(piece["text"]):
            chunks.append({
                "id": f"{meta['doc_id']}::{piece['clause'] or 'pre'}::{len(chunks)}",
                "doc_id": meta["doc_id"], "state": meta["state"], "authority": meta["authority"],
                "title": meta["title"], "date": meta["date"], "status": meta["status"],
                "doc_type": meta["doc_type"], "amends": meta.get("amends") or "",
                "url": meta["source_url"], "clause": piece["clause"], "section": piece["section"],
                "page": piece["page"], "text": part,
                "refs": amended_clauses(part) if meta.get("amends") else [],
            })
    return chunks


def search_text(chunk):
    """The text we search over: a header with the state and document, then the clause text."""
    section = f" ({chunk['section']})" if chunk["section"] else ""
    return (f"{chunk['state']} | {chunk['authority']} | {chunk['title']} | "
            f"Clause {chunk['clause'] or '-'}{section}\n{chunk['text']}")

Now run it over every PDF:

In [6]:
chunks = []
for meta in manifest:
    if meta.get("skip"):
        print("SKIP", meta["file"], "-", meta["skip_reason"])
        continue
    lines = read_pdf_lines(RAW_DIR / meta["file"])
    clauses = split_into_clauses(lines,
                                 is_amendment=meta["doc_type"] in ("amendment", "order"),
                                 needs_dot=meta.get("heading_style") == "dotted")
    doc_chunks = make_chunks(clauses, meta)
    chunks += doc_chunks
    print(f"{meta['file']:<46} {len(doc_chunks):>4} chunks")

print("\nTotal chunks:", len(chunks))

GJ_GERC_NetMetering_Regulations_2016.pdf         48 chunks


GJ_GERC_NetMetering_Consolidated.pdf             49 chunks
GJ_GERC_NetMetering_4thAmendment_2024.pdf        10 chunks


MH_MERC_Rooftop_RE_Regulations_2019.pdf          63 chunks


MH_MERC_Rooftop_1stAmendment_SOR_2023.pdf        45 chunks


MH_MERC_Rooftop_2ndAmendment_SOR_2024.pdf        36 chunks


KA_KERC_Solar_TariffOrder_FY24.pdf               36 chunks
SKIP KA_KERC_Order_2023.pdf - Scanned PDF with no extractable text; needs OCR (Tesseract) to include.
TN_TANGEDCO_PoliciesAdopted.pdf                   2 chunks


TN_TNERC_GISS_Regulations_2024_DRAFT.pdf        119 chunks


RJ_RERC_DRE_Regulations_2021.pdf                 88 chunks


RJ_RERC_SuoMotu_Order_2024.pdf                   14 chunks
IN_MoP_GreenEnergyOpenAccess_Rules_2022.pdf      11 chunks

Total chunks: 521


In [7]:
# Look at one chunk: the Rajasthan 2024 order that raised the net-metering limit
example = [c for c in chunks if c["doc_id"] == "RJ_SUOMOTU_2024" and c["clause"] == "12"][0]
print(search_text(example)[:500])

Rajasthan | RERC | RERC Suo-Motu Order on net metering capacity under DREGS Regulations, 2021 | Clause 12 (In view of above , the Commission under its power conferred under)
12. In view of above , the Commission under its power conferred under
regulation 19 of the RERC DREGS Regulation 2021 “Power to give
directions”, orders that henceforth net metering arrangement shall be
applicable for loads upto one megawatt (01 MW ) or upto the
sanctioned load /contract demand , whichever is lower, for all



## 3. Build the search indexes

- **Vector index (ChromaDB):** finds clauses with similar *meaning*. Built once and saved to `data/chroma_notebook/`.
- **Keyword index (BM25):** finds clauses with the same *words*, which is good for exact numbers like "500 kW". It takes about a second, so it's built every time.

In [8]:
embeddings = OllamaEmbeddings(model=EMBED_MODEL)


def open_store():
    return Chroma(collection_name="regrag_notebook", embedding_function=embeddings,
                  persist_directory=str(CHROMA_DIR), collection_metadata={"hnsw:space": "cosine"})


store = open_store()

# Reuse the saved index if it holds exactly the current chunks; otherwise (re)build it
if set(store.get()["ids"]) != {c["id"] for c in chunks}:
    print("Building the vector index (about 10 minutes on CPU)...")
    store.delete_collection()                     # start from an empty collection
    store = open_store()
    for start in range(0, len(chunks), 64):       # 64 chunks at a time
        batch = chunks[start:start + 64]
        store.add_texts(texts=[search_text(c) for c in batch],
                        metadatas=[{"id": c["id"], "state": c["state"]} for c in batch],
                        ids=[c["id"] for c in batch])
        print(f"  embedded {min(start + 64, len(chunks))}/{len(chunks)}")

print("Chunks in the vector index:", len(store.get()["ids"]))

Chunks in the vector index: 521


In [9]:
STOPWORDS = set("a an and are as at be by for from has have in is it its of on or shall that the "
                "their these this to under was which with".split())


def words(text):
    """Lowercase words and numbers (keeps '6.2' together), without common words."""
    return [w for w in re.findall(r"[a-z0-9]+(?:\.[0-9]+)*", text.lower()) if w not in STOPWORDS]


bm25 = BM25Okapi([words(search_text(c)) for c in chunks])
chunk_by_id = {c["id"]: c for c in chunks}
print("Keyword index ready:", len(chunks), "chunks")

Keyword index ready: 521 chunks


## 4. Understand the question

Find which state the question is about. People don't always say "Gujarat"; they say "GERC" (the regulator),
"Torrent Power" (a power company) or "Ahmedabad" (a city).

In [10]:
STATE_WORDS = {
    "Gujarat": ["gujarat", "gerc", "geda", "dgvcl", "mgvcl", "pgvcl", "ugvcl", "torrent power", "ahmedabad", "surat"],
    "Maharashtra": ["maharashtra", "merc", "msedcl", "mahadiscom", "mumbai", "pune", "nagpur"],
    "Karnataka": ["karnataka", "kerc", "bescom", "mescom", "hescom", "gescom", "bengaluru", "bangalore"],
    "Tamil Nadu": ["tamil nadu", "tamilnadu", "tnerc", "tangedco", "chennai", "coimbatore"],
    "Rajasthan": ["rajasthan", "rerc", "jvvnl", "avvnl", "jdvvnl", "jaipur", "jodhpur"],
}
OTHER_STATES = ["kerala", "punjab", "haryana", "telangana", "andhra pradesh", "uttar pradesh",
                "madhya pradesh", "west bengal", "odisha", "bihar", "delhi", "goa", "assam"]
CENTRAL_WORDS = ["ministry of power", "mnre", "central government", "open access"]


def has_word(text, word):
    """True if 'word' appears as a whole word ('merc' must not match inside 'commercial')."""
    return re.search(rf"(?<![a-z]){re.escape(word)}(?![a-z])", text.lower()) is not None


def find_states(question):
    return [state for state, names in STATE_WORDS.items() if any(has_word(question, n) for n in names)]


def out_of_scope(question):
    """A decline message if the question is only about states we have no documents for."""
    others = [s.title() for s in OTHER_STATES if has_word(question, s)]
    if others and not find_states(question):
        return f"I can't answer this: the documents cover Gujarat, Maharashtra, Karnataka, Tamil Nadu, Rajasthan and central rules - not {', '.join(others)}."
    return None


def question_for_state(question, state):
    """For comparisons: one clean question per state.
    Remove ALL state names, tidy the words left behind, then add "in <state>?" at the end."""
    text = question
    for names in STATE_WORDS.values():
        for name in names:
            text = re.sub(rf"(?i)(?<![a-z]){re.escape(name)}(?![a-z])", "", text)
    text = re.sub(r"(?i)\b(and|versus|vs)\b\.?(?=\s*([?.,]|$))", "", text)    # "...in  and ." -> "...in  ."
    text = re.sub(r"(?i)\b(compare|between)\s+(and|versus|vs)\s+", r"\1 ", text)  # "Compare  and limits"
    text = re.sub(r"(?i)\b(in|for|of)\s*(?=[?.,]|$)", "", text)               # "...solar in ." -> "...solar ."
    text = re.sub(r"\s+", " ", text).strip(" ?.,")
    text = re.sub(r"(?i)^compare\b", "What are", text)                       # "Compare the limits" -> "What are the limits"
    return f"{text} in {state}?"

In [11]:
print(find_states("What does BESCOM pay for rooftop solar exports?"))
print(find_states("Is net metering allowed for commercial consumers in Gujarat?"))   # 'merc' inside 'commercial' is ignored
print(out_of_scope("What is the net metering limit in Kerala?"))
q = "Compare the distribution transformer capacity limits for rooftop solar in Gujarat and Maharashtra."
print(question_for_state(q, "Gujarat"))
print(question_for_state(q, "Maharashtra"))

['Karnataka']
['Gujarat']
I can't answer this: the documents cover Gujarat, Maharashtra, Karnataka, Tamil Nadu, Rajasthan and central rules - not Kerala.
What are the distribution transformer capacity limits for rooftop solar in Gujarat?
What are the distribution transformer capacity limits for rooftop solar in Maharashtra?


## 5. Search: find the best 5 clauses

1. **Two searches, only inside the question's state.** A Rajasthan question can never get a Gujarat clause.
2. **Merge the two lists with reciprocal-rank fusion:** each chunk scores `1 / (60 + rank)` in each list, and the scores are added. Chunks that *both* searches like rise to the top.
3. **Rerank** the best 30 with a cross-encoder. It reads the question and the clause *together*, so it's more accurate, and keep the top 5.

In [12]:
def vector_search(question, states, k=SEARCH_K):
    state_filter = {"state": {"$in": states}} if states else None
    docs = store.similarity_search(question, k=k, filter=state_filter)
    return [doc.metadata["id"] for doc in docs]


def keyword_search(question, states, k=SEARCH_K):
    scores = bm25.get_scores(words(question))
    best_first = sorted(range(len(chunks)), key=lambda i: scores[i], reverse=True)
    result = []
    for i in best_first:
        if scores[i] <= 0 or len(result) == k:
            break
        if states is None or chunks[i]["state"] in states:
            result.append(chunks[i]["id"])
    return result


def hybrid_search(question, states):
    """Merge the two rankings with reciprocal-rank fusion."""
    fused = defaultdict(float)
    for ranking in (vector_search(question, states), keyword_search(question, states)):
        for rank, chunk_id in enumerate(ranking, start=1):
            fused[chunk_id] += 1 / (60 + rank)
    best_first = sorted(fused, key=fused.get, reverse=True)
    return [chunk_by_id[i] for i in best_first]


reranker = CrossEncoder(RERANK_MODEL, max_length=512)


def rerank(question, candidates):
    """Score each (question, chunk) pair and return hits sorted best first."""
    if not candidates:
        return []
    scores = reranker.predict([(question, search_text(c)) for c in candidates])
    hits = [{"chunk": c, "score": float(s), "flags": []} for c, s in zip(candidates, scores)]
    return sorted(hits, key=lambda h: h["score"], reverse=True)


def retrieve(question, states):
    return rerank(question, hybrid_search(question, states)[:30])[:TOP_N]

In [13]:
hits = retrieve("What is the current net metering capacity limit in Rajasthan?", ["Rajasthan"])
for hit in hits:
    c = hit["chunk"]
    print(round(hit["score"], 2), "|", c["doc_id"], "| clause", c["clause"], "|", c["date"])

5.94 | RJ_SUOMOTU_2024 | clause 10 | 2024-02-07
5.73 | RJ_SUOMOTU_2024 | clause 9 | 2024-02-07
5.07 | RJ_SUOMOTU_2024 | clause 6 | 2024-02-07
4.88 | RJ_SUOMOTU_2024 | clause 12 | 2024-02-07
4.74 | RJ_SUOMOTU_2024 | clause 7 | 2024-02-07


## 6. Check for amendments

Rules change. Rajasthan's net-metering limit went from 500 kW (2021) to 1 MW (2024), and Gujarat's 2016 rule
(clause 6.2) was changed in 2017. For each retrieved clause we look for a **later, final** document that changes
that clause. If one is found, the old clause is flagged and moved down, and the newer version is added.

In [14]:
# Which documents amend which (from the manifest), e.g. "GJ_NM_2016" -> ["GJ_NM_CONSOL", "GJ_NM_AMD4_2024"]
amended_by = defaultdict(list)
for meta in manifest:
    if meta.get("amends"):
        amended_by[meta["amends"]].append(meta["doc_id"])


def changes_clause(newer, clause):
    """Does chunk 'newer' change this clause number?"""
    if not clause:
        return False
    if newer["doc_type"] == "consolidated_regulation":       # a rewritten version: same clause number
        return newer["clause"] == clause
    major = clause.split(".")[0]
    # "Regulation 7 substituted" covers 7.3; "Clause 5.1" covers 5.1 and 5.1.2
    return any(ref == clause or ref == major or clause.startswith(ref + ".") for ref in newer["refs"])


def newer_versions(chunk):
    return [c for c in chunks
            if c["doc_id"] in amended_by[chunk["doc_id"]]
            and c["date"] > chunk["date"]
            and c["status"] != "draft"                # a draft never replaces law in force
            and changes_clause(c, chunk["clause"])]


def check_amendments(question, hits):
    present = {h["chunk"]["id"] for h in hits}
    added = []
    for hit in hits:
        chunk = hit["chunk"]
        if chunk["status"] == "draft":
            hit["flags"].append("DRAFT - a proposal, not yet in force")
        newer = newer_versions(chunk)
        if newer:
            latest = max(newer, key=lambda c: c["date"])
            hit["flags"].append(f"SUPERSEDED - clause {chunk['clause']} ({chunk['date']}) was changed by "
                                f"'{latest['title']}' dated {latest['date']}")
            missing = [c for c in newer if c["id"] not in present]
            for new_hit in rerank(question, missing)[:1]:            # add the most relevant newer version
                new_hit["flags"].append(f"AMENDMENT - changes clause {chunk['clause']} of '{chunk['title']}'")
                present.add(new_hit["chunk"]["id"])
                added.append(new_hit)

    def is_old(hit):
        return any(f.startswith(("SUPERSEDED", "DRAFT")) for f in hit["flags"])

    # Current rules first (best score first), then superseded/draft ones
    return sorted(hits + added, key=lambda h: (is_old(h), -h["score"]))

In [15]:
gujarat_question = "Is a residential consumer in Gujarat limited to 50% of sanctioned load for rooftop solar?"
gujarat_hits = check_amendments(gujarat_question, retrieve(gujarat_question, ["Gujarat"]))

for hit in gujarat_hits:
    c = hit["chunk"]
    print(c["doc_id"], "| clause", c["clause"], "|", c["date"])
    for flag in hit["flags"]:
        print("    !", flag[:110])

GJ_NM_CONSOL | clause 6.2 | 2017-10-06
GJ_NM_CONSOL | clause 20 | 2017-10-06
GJ_NM_CONSOL | clause 20 | 2017-10-06
    ! AMENDMENT - changes clause 20 of 'GERC (Net Metering Rooftop Solar PV Grid Interactive Systems) Regulations, 2
GJ_NM_CONSOL | clause 20 | 2017-10-06
    ! AMENDMENT - changes clause 20 of 'GERC (Net Metering Rooftop Solar PV Grid Interactive Systems) Regulations, 2
GJ_NM_2016 | clause 6.2 | 2016-06-18
    ! SUPERSEDED - clause 6.2 (2016-06-18) was changed by 'GERC Net Metering Regulations, 2016 - consolidated with F
GJ_NM_2016 | clause 20 | 2016-06-18
    ! SUPERSEDED - clause 20 (2016-06-18) was changed by 'GERC Net Metering Regulations, 2016 - consolidated with Fi
GJ_NM_2016 | clause 20 | 2016-06-18
    ! SUPERSEDED - clause 20 (2016-06-18) was changed by 'GERC Net Metering Regulations, 2016 - consolidated with Fi


## 7. Write the answer

The LLM gets strict rules: answer **only** from the numbered sources, cite every sentence like `[S2]`, prefer
current rules over superseded ones, and reply `INSUFFICIENT_CONTEXT` if the sources don't answer the question.
If even the best search score is very low, we decline **without** calling the LLM.

In [16]:
llm = ChatGroq(model=LLM_MODEL, temperature=0, max_retries=3)

RULES ="""You are a regulatory analyst assistant for Indian renewable-energy regulations.
Answer the question in 1 to 3 plain sentences, using ONLY the numbered sources. Rules:
1. Base the answer on CURRENT SOURCES. OLDER OR DRAFT SOURCES are for context only; if one says something different, say the earlier rule was changed.
2. End every sentence with its citation, e.g. [S2].
3. Copy numbers (kW, MW, %, Rs, dates) exactly, and name the clause and the order with its date.
4. If the only relevant source is a DRAFT, say it is a proposal, not law in force.
5. If no source answers the question, reply with only: INSUFFICIENT_CONTEXT
6. Never use outside knowledge."""

CANT_ANSWER = "I can't answer this from the indexed regulations - the retrieved clauses don't contain the information."


def format_sources(hits):
    """Number the sources [S1], [S2]... and put superseded/draft ones in a separate section."""
    current, older = [], []
    for number, hit in enumerate(hits, start=1):
        c = hit["chunk"]
        block = f"[S{number}] {c['state']} | {c['title']} | Clause {c['clause'] or '-'} | dated {c['date']} | page {c['page']}"
        for flag in hit["flags"]:
            block += f"\nFLAG: {flag}"
        block += "\n" + c["text"][:1200]
        is_old = any(f.startswith(("SUPERSEDED", "DRAFT")) for f in hit["flags"])
        (older if is_old else current).append(block)
    text = "CURRENT SOURCES:\n\n" + ("\n\n".join(current) or "(none)")
    if older:
        text += "\n\nOLDER OR DRAFT SOURCES (context only):\n\n" + "\n\n".join(older)
    return text


def write_answer(question, hits):
    """Returns {'text', 'declined', 'sources'}."""
    if not hits or max(h["score"] for h in hits) < ABSTAIN_SCORE:
        return {"text": CANT_ANSWER, "declined": True, "sources": []}

    # The question goes before AND after the sources, so the model doesn't lose track of it
    prompt = f"QUESTION: {question}\n\n{format_sources(hits)}\n\nQUESTION (again): {question}\n\nANSWER:"
    reply = llm.invoke([("system", RULES), ("human", prompt)]).content.strip()

    # Tidy the model's special characters: 【S1】 -> [S1], non-breaking hyphens/spaces -> normal ones
    reply = reply.replace("【", "[").replace("】", "]")
    for odd, normal in {"\u2011": "-", "\u2010": "-", "\u202f": " ", "\u00a0": " "}.items():
        reply = reply.replace(odd, normal)

    # Which sources were cited? Handles [S2] and grouped citations like [S2, S3]
    inside_brackets = " ".join(re.findall(r"\[([^\]]*)\]", reply))
    cited = sorted({int(n) for n in re.findall(r"S(\d+)", inside_brackets) if 1 <= int(n) <= len(hits)})

    if "INSUFFICIENT_CONTEXT" in reply.upper().replace(" ", "_"):
        if not cited:
            return {"text": CANT_ANSWER, "declined": True, "sources": []}
        reply = re.sub(r"(?im)^.*INSUFFICIENT[_ ]CONTEXT.*$", "", reply).strip()   # stray token: drop it

    sources = [{"id": f"S{n}", **hits[n - 1]} for n in cited]
    return {"text": reply, "declined": False, "sources": sources}

### ⏱ Answer the Gujarat question

In [17]:
result = write_answer(gujarat_question, gujarat_hits)

print(result["text"])
print()
for s in result["sources"]:
    print(s["id"], "|", s["chunk"]["title"][:60], "| clause", s["chunk"]["clause"], "|", s["chunk"]["date"])

No, a residential consumer in Gujarat is not limited to 50 % of its sanctioned load for rooftop solar; the regulations state that for residential consumers the rooftop solar PV capacity is irrespective of their sanctioned load/contract demand, subject only to a minimum of 1 kW and a maximum of 1 MW[S1][S2].

S1 | GERC Net Metering Regulations, 2016 - consolidated with Firs | clause 6.2 | 2017-10-06
S2 | GERC Net Metering Regulations, 2016 - consolidated with Firs | clause 20 | 2017-10-06


## 8. The full pipeline

One function that runs steps 4–7. For a comparison ("Gujarat and Maharashtra"), each state is searched and
answered separately, then the answers are joined.

In [18]:
def make_plan(question):
    """A list of (state, question) pairs: one per state; (None, question) if no state is mentioned."""
    states = find_states(question)
    if not states:
        return [(None, question)]
    if len(states) == 1:
        return [(states[0], question)]
    return [(state, question_for_state(question, state)) for state in states]


def search_scope(state, question):
    """Which documents to search: the state's own, plus central rules if the question mentions them."""
    if state is None:
        return None
    if any(has_word(question, w) for w in CENTRAL_WORDS):
        return [state, "Central"]
    return [state]


def join_answers(parts):
    """Join per-state answers into one result."""
    if len(parts) == 1:
        return parts[0][1]
    text = "\n\n".join(f"**{state}:** {r['text']}" for state, r in parts)
    sources = [{**s, "id": f"{s['id']}-{state[:2].upper()}"} for state, r in parts for s in r["sources"]]
    return {"text": text, "declined": all(r["declined"] for _, r in parts), "sources": sources}


def answer(question):
    message = out_of_scope(question)
    if message:
        return {"text": message, "declined": True, "sources": []}
    parts = []
    for state, sub_question in make_plan(question):
        hits = retrieve(sub_question, search_scope(state, question))
        hits = check_amendments(sub_question, hits)
        parts.append((state, write_answer(sub_question, hits)))
    return join_answers(parts)


def show(result):
    print(result["text"])
    for s in result["sources"]:
        c = s["chunk"]
        print(f"   [{s['id']}] {c['state']} | {c['title'][:55]} | clause {c['clause']} | {c['date']}")

In [19]:
# Instant: no search, no LLM
show(answer("What is the net metering limit for rooftop solar in Kerala?"))

I can't answer this: the documents cover Gujarat, Maharashtra, Karnataka, Tamil Nadu, Rajasthan and central rules - not Kerala.


### ⏱ A single-state question and a comparison

In [20]:
show(answer("What is the current net metering capacity limit in Rajasthan?"))

The current net-metering capacity limit in Rajasthan is one megawatt (01 MW) per consumer, subject to not exceeding the sanctioned load/contract demand [S4].
   [S4] Rajasthan | RERC Suo-Motu Order on net metering capacity under DREG | clause 12 | 2024-02-07


In [21]:
show(answer("Compare the distribution transformer capacity limits for rooftop solar in Gujarat and Maharashtra."))

**Gujarat:** The cumulative rooftop-solar capacity that may be connected to a given distribution transformer shall not exceed 65 % of the transformer’s peak capacity [ S4 ]. The distribution licensee must update the available transformer capacity on a yearly basis and provide this information to the Commission [ S1 ].

**Maharashtra:** The cumulative capacity of rooftop solar systems (under Net-Metering or Net-Billing) that can be connected to a particular distribution transformer/feeder must not exceed 70 % of the transformer’s rated capacity, although the licensee may allow a higher share after a detailed load-study assessment [S2]. This transformer-wise limit governs the maximum rooftop solar capacity that can be installed at an eligible consumer’s premises, with a permissible variation of up to five percent in the rated capacity [S4].
   [S1-GU] Gujarat | GERC Net Metering Regulations, 2016 - consolidated with | clause 5.1 | 2017-10-06
   [S4-GU] Gujarat | GERC Net Metering Regulat

## 9. The agent (LangGraph)

The same steps as a graph, plus a **self-check**. After answering, the LLM is asked whether the answer is
supported by the sources. If not, it rewrites the question in regulatory wording and tries **once** more,
and it keeps the first answer if the retry is worse.

```
classify → retrieve → check_amendments → write → verify ──ok──▶ END
              ▲                                     │
              └──────────── rewrite ◀── not ok, first try
```

In [22]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph


class AgentState(TypedDict, total=False):
    question: str      # the user's question
    plan: list         # [(state, search question)]
    hits: dict         # state -> hits
    result: dict       # the answer
    previous: dict     # the first answer, kept in case the retry is worse
    verified: bool
    attempts: int


def classify(s):
    message = out_of_scope(s["question"])
    if message:
        return {"result": {"text": message, "declined": True, "sources": []}}
    return {"plan": make_plan(s["question"]), "attempts": 0}


def retrieve_node(s):
    return {"hits": {state: retrieve(q, search_scope(state, s["question"])) for state, q in s["plan"]}}


def amendments_node(s):
    return {"hits": {state: check_amendments(q, s["hits"][state]) for state, q in s["plan"]}}


def write_node(s):
    question = s["question"] if len(s["plan"]) == 1 else None
    parts = [(state, write_answer(question or q, s["hits"][state])) for state, q in s["plan"]]
    result = join_answers(parts)
    if result["declined"] and s.get("previous"):       # the retry got worse: keep the first answer
        result = s["previous"]
    return {"result": result, "attempts": s["attempts"] + 1}


judge = ChatGroq(model=LLM_MODEL, temperature=0, max_retries=3)


def verify(s):
    result = s["result"]
    if result["declined"]:
        return {"verified": s["attempts"] > 1}          # retry once before accepting "can't answer"
    sources = "\n\n".join(format_sources(hits) for hits in s["hits"].values())
    reply = judge.invoke(f"Reply with one word, YES or NO. Is every fact in the ANSWER supported by the SOURCES?"
                         f"\n\nSOURCES:\n{sources}\n\nANSWER:\n{result['text']}").content
    return {"verified": reply.strip().upper().startswith("YES")}


rewriter = ChatGroq(model=LLM_MODEL, temperature=0, max_retries=3)


def rewrite(s):
    new_question = rewriter.invoke(
        "Rewrite this question in the wording of Indian electricity regulations (net metering, sanctioned load, "
        "contract demand, banking...). Keep the state names. Reply with the question only.\n\n" + s["question"]
    ).content.strip().splitlines()[0]
    plan = [(state, new_question if state is None or state.lower() in new_question.lower() else f"{new_question} in {state}")
            for state, _ in s["plan"]]
    print("   (rewritten:", new_question, ")")
    return {"plan": plan, "previous": s["result"]}


graph_builder = StateGraph(AgentState)
graph_builder.add_node("classify", classify)
graph_builder.add_node("retrieve", retrieve_node)
graph_builder.add_node("check_amendments", amendments_node)
graph_builder.add_node("write", write_node)
graph_builder.add_node("verify", verify)
graph_builder.add_node("rewrite", rewrite)

def after_classify(s):
    return "done" if s.get("result") else "retrieve"      # out of scope: stop straight away


def after_verify(s):
    return "done" if s["verified"] or s["attempts"] > 1 else "rewrite"   # retry only once


graph_builder.add_edge(START, "classify")
graph_builder.add_conditional_edges("classify", after_classify, {"done": END, "retrieve": "retrieve"})
graph_builder.add_edge("retrieve", "check_amendments")
graph_builder.add_edge("check_amendments", "write")
graph_builder.add_edge("write", "verify")
graph_builder.add_conditional_edges("verify", after_verify, {"done": END, "rewrite": "rewrite"})
graph_builder.add_edge("rewrite", "retrieve")

agent = graph_builder.compile()
print(agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	classify(classify)
	retrieve(retrieve)
	check_amendments(check_amendments)
	write(write)
	verify(verify)
	rewrite(rewrite)
	__end__([<p>__end__</p>]):::last
	__start__ --> classify;
	check_amendments --> write;
	classify -. &nbsp;done&nbsp; .-> __end__;
	classify -.-> retrieve;
	retrieve --> check_amendments;
	rewrite --> retrieve;
	verify -. &nbsp;done&nbsp; .-> __end__;
	verify -.-> rewrite;
	write --> verify;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### ⏱ Run the agent

In [23]:
final_state = agent.invoke({"question": "What is the current net metering capacity limit in Rajasthan?"})

print("Verified:", final_state["verified"], "| attempts:", final_state["attempts"])
print()
show(final_state["result"])

Verified: True | attempts: 1

The current net-metering capacity limit in Rajasthan is one megawatt (01 MW) per consumer, subject to not exceeding the sanctioned load or contract demand [ S4 ][ S1 ].
   [S1] Rajasthan | RERC Suo-Motu Order on net metering capacity under DREG | clause 10 | 2024-02-07
   [S4] Rajasthan | RERC Suo-Motu Order on net metering capacity under DREG | clause 12 | 2024-02-07


## 10. Evaluation

We test the pipeline on **34 hand-written questions** (`eval/testset.json`). Each reference answer was
checked against the PDFs. The questions cover:

| Type | What it tests |
|---|---|
| `single` | A fact from one state's regulation |
| `amendment` | A rule that a later order changed: does the answer use the current version? |
| `draft` | A draft regulation: is it flagged as a proposal? |
| `central` | National rules (Ministry of Power) |
| `multi` | A two-state comparison |
| `abstain` | A question the documents can't answer: the system should decline |

**What we measure.** This runs live with Groq, about 4 minutes:
- **Retrieval hit:** did a clause from the expected document reach the model?
- **Contamination:** did any clause from a state *not* in the question reach the model? It should be 0%.
- **Decline accuracy:** did it answer the answerable questions and decline the rest?
- **Citation rate:** do answers cite their sources?
- **Key-number recall:** share of the reference answer's numbers (`1 MW`, `65%`, `Rs 4.50`…) that appear in the answer.

In [24]:
import time
import pandas as pd

testset = json.loads((ROOT / "eval" / "testset.json").read_text(encoding="utf-8"))
print(len(testset), "test questions:", dict(Counter(t["type"] for t in testset)))
print()
print("Example:", testset[16]["question"])
print("Reference:", testset[16]["ground_truth"])

34 test questions: {'single': 20, 'amendment': 6, 'draft': 2, 'central': 2, 'multi': 2, 'abstain': 2}

Example: What is the current net metering capacity limit in Rajasthan?
Reference: By RERC suo-motu order dated 07.02.2024, net metering applies to loads up to one megawatt (1 MW) or up to the sanctioned load/contract demand, whichever is lower, for all consumer categories - raised from the earlier limit of 500 kW set by the suo-motu order of 07.09.2021.


In [25]:
def run_question(question):
    """Same as answer(), but also returns every retrieved hit so retrieval can be scored too."""
    message = out_of_scope(question)
    if message:
        return {"text": message, "declined": True, "sources": []}, []
    parts, all_hits = [], []
    for state, sub_question in make_plan(question):
        hits = check_amendments(sub_question, retrieve(sub_question, search_scope(state, question)))
        all_hits += hits
        parts.append((state, write_answer(sub_question, hits)))
    return join_answers(parts), all_hits


NUMBER_WORDS = {"one": "1", "thirty": "30", "fifty": "50", "five hundred": "500"}


def key_numbers(text):
    """The figures in a text, normalised: 'one megawatt (01 MW)' -> {'1mw'}, 'Rs. 10,000' -> {'rs10000'}."""
    t = text.lower().replace(",", "")
    for word, digit in NUMBER_WORDS.items():                       # "thirty percent" -> "30 percent"
        t = re.sub(rf"\b{word}\s+(?=mega|mw|per)", digit + " ", t)
    t = t.replace("megawatt", "mw").replace("kilowatt", "kw").replace("per cent", "%").replace("percent", "%")
    t = re.sub(r"\b0+(\d)", r"\1", t)                             # "01 mw" -> "1 mw"
    found = re.findall(r"(?:rs\.?|inr)\s?\d+(?:\.\d+)?|\d+(?:\.\d+)?\s?(?:%|kw|mw|kva|days?|years?)(?![a-z])", t)
    cleaned = set()
    for f in found:
        f = f.replace(" ", "").replace("rs.", "rs").replace("inr", "rs")
        cleaned.add(re.sub(r"(day|year)s$", r"\1", f))
    return cleaned


def score(test, result, hits):
    """All the measures for one test question."""
    should_decline = test["type"] == "abstain"
    expected = key_numbers(test["ground_truth"]) if not should_decline else set()
    allowed_states = set(test["states"]) | {"Central"}
    return {
        "id": test["id"],
        "type": test["type"],
        "decline_correct": result["declined"] == should_decline,
        "cited": bool(result["sources"]) if not result["declined"] else None,
        "retrieval_hit": (any(h["chunk"]["doc_id"] in test["expected_docs"] for h in hits)
                          if test["expected_docs"] else None),
        "wrong_state_hits": sum(h["chunk"]["state"] not in allowed_states for h in hits) if test["states"] else 0,
        "hits": len(hits),
        "numbers_expected": len(expected),
        "numbers_found": len(expected & key_numbers(result["text"])),
        "answer": result["text"],
    }


print(key_numbers("up to one megawatt (01 MW), 65 % of peak, Rs. 10,000 and thirty percent"))

{'1mw', '30%', '65%', 'rs10000'}


### ⏱ Run all 34 questions (about 4 minutes)

In [26]:
rows = []
start = time.time()
for i, test in enumerate(testset, start=1):
    result, hits = run_question(test["question"])
    rows.append(score(test, result, hits))
    print(f"{i:>2}/{len(testset)} {test['id']:<5} declined={result['declined']!s:<5}  "
          f"numbers {rows[-1]['numbers_found']}/{rows[-1]['numbers_expected']}")
    time.sleep(1)   # stay gentle with Groq's free-tier rate limit
print(f"\nDone in {time.time() - start:.0f}s")

 1/34 GJ01  declined=False  numbers 2/2


 2/34 GJ02  declined=False  numbers 3/3


 3/34 GJ03  declined=False  numbers 0/0


 4/34 GJ04  declined=False  numbers 0/0


 5/34 GJ05  declined=False  numbers 0/0


...

31/34 MS01  declined=False  numbers 2/2


32/34 MS02  declined=False  numbers 2/3


33/34 AB01  declined=True   numbers 0/0


34/34 AB02  declined=True   numbers 0/0



Done in 543s


In [27]:
results = pd.DataFrame(rows)
answered = results[results["type"] != "abstain"]

summary = {
    "Retrieval hit (expected document reached the model)": results["retrieval_hit"].dropna().astype(bool).mean(),
    "Contamination (clauses from a wrong state)": results["wrong_state_hits"].sum() / results["hits"].sum(),
    "Decline accuracy (answer vs decline correct)": results["decline_correct"].mean(),
    "Citation rate (answers that cite a source)": results["cited"].dropna().astype(bool).mean(),
    "Key-number recall": answered["numbers_found"].sum() / answered["numbers_expected"].sum(),
}
pd.Series(summary, name="score").map("{:.1%}".format).to_frame()

,score
Retrieval hit (expected document reached the model),100.0%
Contamination (clauses from a wrong state),0.0%
Decline accuracy (answer vs decline correct),97.1%
Citation rate (answers that cite a source),100.0%
Key-number recall,64.0%


In [28]:
# The same measures by question type
by_type = results.groupby("type").agg(
    questions=("id", "count"),
    retrieval_hit=("retrieval_hit", lambda s: s.dropna().astype(bool).mean()),
    decline_correct=("decline_correct", "mean"),
    numbers_found=("numbers_found", "sum"),
    numbers_expected=("numbers_expected", "sum"),
)
by_type["key_number_recall"] = by_type["numbers_found"] / by_type["numbers_expected"]
by_type.round(2)

,questions,retrieval_hit,decline_correct,numbers_found,numbers_expected,key_number_recall
type,,,,,,
abstain,2,NaN,1.00,0,0,NaN
amendment,6,1.0,1.00,4,10,0.40
central,2,1.0,1.00,2,2,1.00
draft,2,1.0,1.00,2,2,1.00
multi,2,1.0,1.00,4,5,0.80
single,20,1.0,0.95,20,31,0.65


In [29]:
# Questions where a reference number is missing or the decline decision was wrong
misses = results[(results["numbers_found"] < results["numbers_expected"]) | ~results["decline_correct"]]
print(len(misses), "questions to look at:\n")
for _, row in misses.iterrows():
    test = next(t for t in testset if t["id"] == row["id"])
    print(f"[{row['id']}] {test['question']}")
    print(f"   reference: {test['ground_truth'][:230]}")
    print(f"   answer:    {row['answer'][:230]}")
    print(f"   numbers found {row['numbers_found']}/{row['numbers_expected']}, decline correct: {row['decline_correct']}\n")

11 questions to look at:

[GJ06] What connectivity charges apply to a 50 kW rooftop solar system in Gujarat?
   reference: Under Annexure-III of the GERC Net Metering Regulations, 2016, systems of more than 6 kW up to 100 kW pay INR 10,000 per connection (1-6 kW: INR 1,500; more than 100 kW up to 1 MW: INR 50,000).
   answer:    A rooftop solar system of 50 kW falls in the “more than 6 kW up to 100 kW” bracket and attracts a connectivity charge of INR 10,000 per connection payable to the distribution licensee [S5].
   numbers found 3/6, decline correct: True

[GJ08] Who bears the cost of distribution system strengthening for a 5 kW rooftop solar system in Gujarat after the 2024 amendment?
   reference: Under clause 5.1 as substituted by the GERC Fourth Amendment Regulations, 2024 (dated 04.09.2024), the cost of strengthening distribution infrastructure for rooftop systems up to 6 kW is included in the distribution licensee's Ann
   answer:    The cost of strengthening the distribution 

### Reading the results

**Retrieval is essentially perfect:** the expected document reached the model for every answerable
question, and no clause from another state ever did (0% contamination).

**Key-number recall (64%) understates answer quality.** It counts a miss whenever a number in the
reference answer is absent, even when the reference lists extra details the question didn't ask for.
Reading the 11 flagged answers by hand:

- **8 are correct answers:**
  - GJ06 gives the right fee (Rs 10,000) without the other fee slabs.
  - RJ01 gives 1 MW without the old 500 kW limit.
  - MH01 omits the 1 kW minimum.
  - TN01 omits the HT capacity range.
  - KA02, GJ08, GJ09 and MS02 omit side details.
- **3 are real errors:**
  - **MH05** declined, although the registration fee is in Regulation 9.1: retrieval ranked the fee table too low.
  - **MH03** points to clause 11.4(c) without saying what it says.
  - **MH04** reaches the right conclusion (small consumers are exempt) for the wrong reason: it quotes a 5,000 MW
    rule about wheeling charges instead of the 10 kW exemption.

**By hand, 31 of 34 questions (91%) are answered correctly**, including both two-state comparisons, all
amendment questions and both questions that should be declined. The remaining errors come from ranking
(MH05) and from answers that cite the right clause without spelling out its content (MH03, MH04).

*Because the LLM's wording can vary between runs, the exact list of flagged answers may differ slightly when you re-run.*

### RAGAS scores (from the earlier full evaluation)

Running RAGAS needs an LLM judge for every claim, which costs several hundred thousand Groq tokens for
34 questions (more than the free daily quota). So the RAGAS scores below are the **saved results** from
`eval/run_eval.py`. Note that they were measured when the answers came from the **local `llama3.2` (3B)**
model, before switching answers to Groq. The judge was Groq `gpt-oss-120b`.

In [30]:
saved = json.loads((ROOT / "eval" / "results" / "ragas_pipeline.json").read_text(encoding="utf-8"))
pd.Series({
    "Faithfulness (claims supported by the sources)": saved["faithfulness"],
    "Context precision (relevant clauses ranked high)": saved["llm_context_precision_with_reference"],
    "Answer relevancy (answer addresses the question)": saved["answer_relevancy"],
}, name=f"RAGAS, judge: {saved['judge']}").to_frame()

,"RAGAS, judge: groq:openai/gpt-oss-120b"
Faithfulness (claims supported by the sources),0.640
Context precision (relevant clauses ranked high),0.789
Answer relevancy (answer addresses the question),0.803


## Summary

| Step | Function(s) |
|---|---|
| Read & split PDFs | `read_pdf_lines`, `split_into_clauses`, `make_chunks` |
| Indexes | `store` (ChromaDB), `bm25` |
| Understand | `find_states`, `out_of_scope`, `question_for_state` |
| Search | `vector_search`, `keyword_search`, `hybrid_search`, `rerank`, `retrieve` |
| Amendments | `newer_versions`, `check_amendments` |
| Answer | `format_sources`, `write_answer` |
| Pipeline | `answer` |
| Agent | `agent` (LangGraph) |
| Evaluation | `run_question`, `key_numbers`, `score` |

The packaged version of the same logic is in `regrag/`, and the web app (`app.py`) uses it.